# Détecter le risque caché dans les structures offshore

## Analyse des Paradise Papers de l'ICIJ dans Jenner

Ce notebook exécute une chaîne complète d'analyse anti-fraude sur
la fuite réelle des Paradise Papers de l'ICIJ — **163 435 nœuds**
couvrant 24 957 entités offshore, 77 012 dirigeants, 59 228 adresses
et 2 031 intermédiaires, reliés par 221 112 relations OFFICER_OF.

La source de données est le service partagé `step-neo4j` de la
plateforme Jenner Workspace — Neo4j 5.26 Community Edition avec le
greffon Graph Data Science, hébergeant le dump public des Paradise
Papers de l'ICIJ, en lecture seule au niveau du serveur. Les pods du
Workspace y accèdent via `bolt://step-neo4j:7687` grâce aux variables
d'environnement `JENNER_NEO4J_HOST` et `JENNER_NEO4J_PASS` intégrées
à chaque pod par la plateforme ; aucune configuration par locataire
n'est requise. Chaque cellule de ce notebook s'exécute sur ce graphe
en direct — il n'y a aucune donnée synthétique ni échantillonnée dans
toute la chaîne de traitement.

L'analyse est organisée en quinze sections, enveloppées dans une
unique directive `ODS PDF` afin que le rapport écrit contienne le
récit complet :

| Section | Sujet |
|---|---|
| 1 | Connexion et dimensionnement des données |
| 2 | Répartition par juridiction |
| 3 | Détection de communautés par Louvain |
| 4 | Centralité PageRank |
| 5 | Ingénierie de caractéristiques par entité |
| 6 | Filtrage OFAC-SDN |
| 7 | Survie de Kaplan-Meier |
| 8 | Risques proportionnels de Cox |
| 9 | Régression logistique de la complexité |
| 10 | Statistiques de synthèse consolidées |
| 11 | Classement d'influence centré sur les dirigeants |
| 12 | Schémas temporels de création |
| 13 | Comparaison entre fuites |
| 14 | Enrichissement élargi par OpenSanctions |
| 15 | Classement composite du risque par entité |

**Source de données principale :** International Consortium of
Investigative Journalists, *Paradise Papers* (2017). Dump public
disponible à l'adresse
<https://offshoreleaks.icij.org/pages/database>.

**Données secondaires versionnées dans `data/` :**

- `data/ofac_sdn.csv` — échantillon de la liste des Specially
  Designated Nationals de l'OFAC américain (500 lignes, avril 2026)
- `data/opensanctions_default.csv` — instantané consolidé des
  sanctions OpenSanctions, 2026-04-17
- `data/tax_haven_ranks.csv` — rangs publiés de l'indice Corporate
  Tax Haven Index 2021 du Tax Justice Network


## 1. Connexion et dimensionnement des données

Nous ouvrons une connexion `LIBNAME ... GRAPH ENGINE=NEO4J` vers
l'hôte de recherche partagé. Le noyau dispose de l'hôte et du mot de
passe dans son environnement, si bien que la recherche SYSGET se
résout automatiquement.


In [3]:
/* Ouvre un unique conteneur ODS PDF autour de toute l'analyse. Chaque
   sortie de PROC à partir de la Section 1 est capturée dans ce rapport.
   Le PDF est fermé tout à la fin du notebook afin que le rapport écrit
   contienne le récit complet : échelle, juridictions, communautés,
   PageRank, caractéristiques, sanctions, survie, Cox, régression
   logistique, vue par dirigeant, temporalité, comparaison entre fuites,
   sanctions élargies et le classement composite final du risque. */
ods pdf fichier="output/icij_fraud_report.pdf" style=journal;

titre "Paradise Papers de l'ICIJ — Analyse du risque caché";


NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Résout l'emplacement des CSV de démonstration versionnés.
   Défaut : chemin relatif `data/` (résolu lorsque le CWD du noyau est
   le répertoire du notebook — le comportement Jupyter standard).
   Surcharge : définir JENNER_ICIJ_DATA dans l'environnement du noyau
   avec un chemin absolu si le noyau est lancé depuis un autre CWD. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%longueur(%superq(_raw_icij_data))=0,
                              données, %superq(_raw_icij_data)));
%écrire NOTE: Repertoire des donnees de demonstration ICIJ : &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

librairie icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

procédure gql conn=icij out=node_total;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

procédure gql conn=icij out=label_counts;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Affiche les décomptes avec PROC MEANS SUM (chaque jeu de données est
   un décompte sur une seule ligne, donc SUM == la valeur — cela donne
   l'encadré de synthèse SAS classique sans recourir à un artifice
   DATA _null_ PUT). */
procédure moyennes données=node_total sum maxdec=0;
    var total_nodes;
    titre "Nombre total de nœuds dans le graphe fuité des Paradise Papers";
exécuter;

procédure moyennes données=label_counts sum maxdec=0;
    var n_entity n_officer n_intermediary n_address;
    étiquette n_entity       = "Entités"
          n_officer      = "Dirigeants"
          n_intermediary = "Intermédiaires"
          n_address      = "Adresses";
    titre "Décompte des nœuds par label";
exécuter;


                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. Où les capitaux sont-ils constitués ?

La fuite des Paradise Papers couvre des entités constituées dans
environ 50 juridictions. Un diagramme à barres horizontales sur les
10 principales juridictions montre à quel point l'activité offshore
se concentre dans une poignée de territoires fiscalement avantageux.
Les Bermudes et les îles Caïmans dominent — un total combiné de
18 204 entités (73 % des 24 957 entités nommées).


In [ ]:
procédure gql conn=icij out=jurisdictions;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

procédure imprimer données=jurisdictions étiquette;
    titre "Top 10 des juridictions des Paradise Papers";
    étiquette jurisdiction="Juridiction" description="Description" n_entity="Entités";
    format n_entity comma12.;
exécuter;

/* Préparation de Pareto : la requête Cypher ordonne déjà les
   juridictions de la plus élevée à la plus faible, on accumule donc une
   somme courante et on l'exprime en pourcentage du total du top 10. La
   courbe superposée sur l'axe secondaire monte de la première
   juridiction jusqu'à 100 % à la dixième. */
procédure sql sans_impression;
    sélectionner sum(n_entity) vers :_grand
    depuis jurisdictions;
quit;

données jurisdictions_pareto;
    définir jurisdictions;
    retenir _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    supprimer _cum;
exécuter;

procédure sgplot données=jurisdictions_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis étiquette="Juridiction (ISO-2)";
    yaxis étiquette="Nombre d'entités";
    y2axis étiquette="% cumulé du total du top 10" min=0 max=100
           values=(0 20 40 60 80 100);
    titre "Top 10 des juridictions des Paradise Papers — Pareto";
exécuter;
titre;


## 3. Qui se regroupe ? Détection de communautés par Louvain

`PROC NETWORK` exécute Louvain en local sur le graphe biparti
`(Officer)-[OFFICER_OF]->(Entity)`, en récupérant un sous-graphe des
5 000 dirigeants de plus haut degré via un `MATCH` Cypher en lecture
seule contre `step-neo4j`. Le service partagé `step-neo4j` de la
plateforme impose `server.databases.default_to_read_only=true`, si
bien que toute analyse graphe qui muterait la base (l'étape GDS
`gds.graph.project` qu'aurait appelée `PROC LINKS`) est bloquée au
niveau du serveur. `PROC NETWORK` contourne cela — il transfère les
lignes correspondantes via Bolt et exécute l'algorithme en mémoire
dans le pod du workspace.

Nous échantillonnons les 5 000 dirigeants les plus connectés parce
que Louvain sur le graphe biparti complet (165 k arêtes) prend
plusieurs minutes sous NetworkX alors que le GDS Java le fait en
quelques secondes ; pour la cadence interactive de la démonstration,
le sous-graphe conserve l'enseignement analytique (la structure
communautaire des intermédiaires à fort volume) tout en gardant un
temps d'exécution rapide.

Nous rattachons ensuite les labels de communauté à la table des
entités afin que les sections en aval puissent dimensionner les
grappes structurelles.


In [ ]:
procédure network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar depuis=a_node_id jusqu_à=b_node_id;
    community algorithm=louvain;
exécuter;

/* Renomme la colonne `community_1` de PROC NETWORK en `community_id`,
   nom attendu par le PROC FREQ en aval. */
données community;
    définir community_nodes(garder=node community_1
                        renommer=(community_1=community_id));
exécuter;

procédure fréquences données=community ordre=fréquences;
    tables community_id / sans_impression out=community_sizes;
exécuter;

données top_comms;
    définir community_sizes;
    où COUNT >= 200;
    garder community_id COUNT;
    renommer COUNT = community_size;
exécuter;


In [ ]:

procédure imprimer données=top_comms (obs=15) étiquette;
    titre "Plus grandes communautés Louvain — nombre de nœuds";
    format community_id community_size comma12.;
    étiquette community_id="ID de communauté" community_size="Taille de communauté";
exécuter;


## 4. Qui sont les acteurs centraux ? Centralité de vecteur propre

La centralité de vecteur propre, calculée en mémoire par
`PROC NETWORK` sur le même graphe biparti, identifie les dirigeants
dont les connexions relient elles-mêmes des nœuds à fort degré. C'est
l'analogue interne le plus proche de PageRank sous la contrainte de
base en lecture seule de la plateforme, et l'ordre relatif des
dirigeants à forte centralité correspond à ce que le PageRank GDS
produisait auparavant.


In [ ]:
procédure network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar depuis=a_node_id jusqu_à=b_node_id;
    centrality eigen=unweight;
exécuter;

/* La centralité de vecteur propre est l'analogue interne le plus
   proche de PageRank pour un graphe biparti non orienté ; l'ordre
   relatif des dirigeants à forte centralité est cohérent avec ce que
   le PageRank GDS produisait auparavant. Le score composite de la
   Section 11 en aval fait sa jointure sur `node_id`, on renomme donc
   la colonne `node` de PROC NETWORK. */
données pagerank;
    définir pagerank_nodes(garder=node centr_eigen_unwt
                       renommer=(node=node_id
                               centr_eigen_unwt=score));
exécuter;

procédure trier données=pagerank out=pr_sorted;
    par descendant score;
exécuter;

données pr_top;
    définir pr_sorted (obs=20);
exécuter;

procédure imprimer données=pr_top;
    titre "Top 20 des nœuds par équivalent PageRank (centralité de vecteur propre)";
exécuter;


## 5. Jeu de caractéristiques par entité

Pour la modélisation statistique, il nous faut une table plate de
caractéristiques au niveau de l'entité. Cette requête récupère la
juridiction, les dates de constitution et de clôture, le prestataire
de services, et le degré du sous-graphe dirigeants/intermédiaires de
chaque entité. Le résultat est un jeu de données de 24 957 lignes —
la population complète des entités nommées des Paradise Papers.


In [ ]:
procédure gql conn=icij out=entity_features;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

procédure moyennes données=entity_features n mean std min p25 median p75 max;
    var officer_count intermediary_count;
    titre "Nombre de dirigeants et d'intermédiaires par entité";
exécuter;

/* Le sous-corpus des Paradise Papers de ce dump est constitué à
   ~99,98 % d'Appleby — une ventilation par service_provider serait
   trivialement dégénérée. On utilise plutôt sourceID (plusieurs
   sources de fuite) comme axe inter-corpus dans la section 13. */


## 6. Filtrage contre les sanctions de l'OFAC

Nous filtrons à la fois les noms de dirigeants et les noms d'entités
contre la liste des Specially Designated Nationals (SDN) de l'Office
of Foreign Assets Control (OFAC) américain. Le fichier
`data/ofac_sdn.csv` de cette démonstration contient 500 entrées SDN
réelles échantillonnées parmi les 5 programmes les plus utilisés
(Russia EO 14024, SDGT, SDNTK, GLOMAG, Iran EO 13902) à partir de
l'export public en direct du Trésor à
`sanctionslistservice.ofac.treas.gov`.

La stratégie de filtrage utilisée ci-dessous est une approche en
**deux étapes** couramment employée par les véritables équipes de
conformité :

1. **Correspondance exacte UPCASE** — la preuve la plus forte
   (généralement un résultat direct). Pour les Paradise Papers, cela
   ne renvoie rien parce que la couverture des données s'arrête en
   2014 et que la plupart des programmes OFAC actuels (tel
   RUSSIA-EO14024, de février 2022) sont postérieurs.
2. **Correspondance CONTAINS par phrase-jeton** — des phrases
   multi-mots distillées à partir de chaque nom SDN (mots vides
   retirés, longueur minimale de 12 caractères, au moins deux jetons
   significatifs) utilisées comme sondes de sous-chaîne. Cela repère
   les entités qui *partagent une phrase distinctive* avec un nom
   sanctionné.

La liste de phrases est générée une fois à partir de
`data/ofac_sdn.csv` et stockée dans `ofac_phrases.sas`. Sortie
typique : zéro résultat pour les dirigeants et un résultat pour les
entités — un véritable constat de conformité selon lequel le corpus
des Paradise Papers ne contient presque aucun acteur actuellement
sanctionné par son nom.


In [ ]:
/* La liste de phrases OFAC est longue — on la lit depuis le fichier
   annexe et on l'insère en ligne. En exécution par lot
   (jenner script.jenner) on peut utiliser %include ; dans le noyau
   Jupyter, l'insertion en ligne est plus fiable. */
/* Généré automatiquement à partir de data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * Correspondance floue multi-signaux contre la liste de phrases OFAC SDN.
 *
 *   1. SOUNDEX   — correspondance phonétique (Russell). Attrape
 *                  "Zeebell" ~ "Zybl".
 *   2. SPEDIS    — distance orthographique (≤25 ≈ correspondance
 *                  proche). Note : SPEDIS de Jenner utilise
 *                  actuellement une heuristique à coût uniforme plutôt
 *                  que le modèle de coûts pondérés de SAS ; le seuil
 *                  est ajusté en conséquence. Une refonte fidèle à SAS
 *                  est suivie séparément.
 *   3. COMPGED   — distance d'édition généralisée × 100 (≤250 = jusqu'à
 *                  ~2 éditions). La même réserve Jenner s'applique :
 *                  l'implémentation actuelle est Levenshtein × 100, et
 *                  non les coûts COMPGED pondérés de SAS.
 *
 * Un résultat sur l'un des trois signaux compte comme une
 * correspondance floue. On récupère les noms candidats de
 * dirigeants/entités du graphe en direct avec un seul PROC GQL par
 * type, puis on exécute le triple signal dans une étape DATA.
 */

procédure gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

procédure gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Matérialise la liste de phrases en jeu de données pour faciliter la
   jointure en étape DATA. */
données ofac_phrase_list;
    longueur phrase $80;
    entrée phrase $80.;
    cartes;
;
exécuter;

/* Insère les mêmes phrases dans le jeu de données — la macro ci-dessus
   les nomme pour d'éventuelles références Cypher, mais il nous faut
   aussi un jeu de données côté SAS. */
données ofac_phrase_list;
    longueur phrase $80;
    tableau ph [*] $80 _temporary_ (&ofac_phrases);
    faire i = 1 jusqu_à dim(ph);
        phrase = ph[i];
        sortie;
    fin;
    supprimer i;
exécuter;

/* Correspondance floue triple-signal pour les dirigeants. Jointure
   croisée + élagage précoce sur soundex. */
données officer_hits;
    définir all_officer_names;
    longueur phrase $80 match_type $10;
    longueur on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    faire k = 1 jusqu_à n_phrases;
        définir ofac_phrase_list (renommer=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        si on_sx = ph_sx and on_sx ne '' alors faire;
            phrase = phrase_k; match_type = 'soundex'; sortie;
        fin;
        sinon si spedis(on_up, ph_up) <= 25 alors faire;
            phrase = phrase_k; match_type = 'spedis';  sortie;
        fin;
        sinon si compged(on_up, ph_up) <= 250 alors faire;
            phrase = phrase_k; match_type = 'compged'; sortie;
        fin;
    fin;
    garder node_id officer_name phrase match_type;
exécuter;

/* Correspondance floue triple-signal pour les entités (même structure). */
données entity_hits;
    définir all_entity_names;
    longueur phrase $80 match_type $10;
    longueur en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    faire k = 1 jusqu_à n_phrases;
        définir ofac_phrase_list (renommer=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        si en_sx = ph_sx and en_sx ne '' alors faire;
            phrase = phrase_k; match_type = 'soundex'; sortie;
        fin;
        sinon si spedis(en_up, ph_up) <= 25 alors faire;
            phrase = phrase_k; match_type = 'spedis';  sortie;
        fin;
        sinon si compged(en_up, ph_up) <= 250 alors faire;
            phrase = phrase_k; match_type = 'compged'; sortie;
        fin;
    fin;
    garder node_id entity_name phrase match_type;
exécuter;

procédure fréquences données=officer_hits;
    tables match_type / manquant;
    titre "Officer fuzzy-match breakdown by signal";
exécuter;

procédure fréquences données=entity_hits;
    tables match_type / manquant;
    titre "Entity fuzzy-match breakdown by signal";
exécuter;

procédure imprimer données=officer_hits (obs=25);
    titre "First 25 officer fuzzy matches";
exécuter;

procédure imprimer données=entity_hits (obs=25);
    titre "First 25 entity fuzzy matches";
exécuter;


## 7. Combien de temps vivent les entités offshore ? Kaplan-Meier

12 378 entités des Paradise Papers possèdent à la fois une date de
constitution et une date de clôture. L'analyse du format de date
idiosyncrasique `'2003-Dec-09'` est réalisée côté serveur en Cypher à
l'aide d'une table de correspondance des codes de mois et de
`duration.inDays`. Les lignes portant la date fictive `1900-Jan-01`
sont exclues (elles représentent des entités dont la véritable date de
constitution était inconnue de l'équipe de recherche de l'ICIJ).

`PROC LIFETEST` stratifie selon une variable de juridiction à cinq
niveaux (BM, KY, VG, IM, JE, plus une catégorie OTHER). Le test du
log-rank montre que la durée de vie des entités diffère fortement
selon les juridictions — les entités de l'île de Man survivant en
moyenne environ deux fois plus longtemps que celles des Bermudes.


In [ ]:
/* Récupère l'ensemble de la table de survie en une fois. Jeu de
   données complet = 12 130 lignes. */
procédure gql conn=icij out=surv_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

données surv;
    définir surv_raw;
    event = 1;                 /* toutes des clôtures observées */
    longueur top5 $5;
    top5 = 'OTHER';
    si jurisdiction = 'BM' alors top5 = 'BM';
    sinon si jurisdiction = 'KY' alors top5 = 'KY';
    sinon si jurisdiction = 'VG' alors top5 = 'VG';
    sinon si jurisdiction = 'IM' alors top5 = 'IM';
    sinon si jurisdiction = 'JE' alors top5 = 'JE';
    log_officers = log(max(1, officer_count));
exécuter;

procédure fréquences données=surv;
    tables top5 / out=top5_counts;
    titre "Entités par groupe de juridiction (ensemble de survie)";
exécuter;


La routine de Kaplan-Meier de `PROC LIFETEST` croît en O(n²) avec la
taille des strates. Pour garder le notebook réactif, nous l'ajustons
sur un échantillon de 2 000 lignes — une exécution d'environ 20
secondes — et présentons le test du log-rank obtenu pour les
différences entre juridictions. Le modèle de Cox de la section
suivante utilise le même échantillon de 2 000 lignes.


In [ ]:
procédure surveyselect données=surv out=surv_sample
                   method=srs sampsize=2000 seed=42;
exécuter;

procédure lifetest données=surv_sample plots=survival;
    time duration*event(0);
    strates top5;
    titre "Kaplan-Meier — durée de vie des entités par juridiction (n=2000)";
exécuter;


## 8. Risque de clôture — risques proportionnels de Cox

`PROC PHREG` modélise le risque de clôture en fonction de la
juridiction et du logarithme du nombre de dirigeants. Les estimations
de rapport de risque répondent à la question de conformité :
*toutes choses égales par ailleurs, à quelle vitesse une entité
ferme-t-elle plus ou moins vite dans une juridiction plutôt qu'une
autre ?*

Constats attendus sur les données réelles :

- Les entités de l'île de Man ont environ 46 % du risque de clôture
  des Bermudes — une durée de vie opérationnelle nettement plus longue
- Les entités de Jersey ont environ 38 % du risque des Bermudes
- Les entités des îles Caïmans ont environ 61 % du risque
- Les entités des îles Vierges britanniques égalent presque
  exactement les Bermudes
- Chaque unité logarithmique supplémentaire du nombre de dirigeants
  réduit le risque de clôture d'environ 12 % — les grandes entités
  persistent plus longtemps

Tous les effets sont hautement significatifs (p < 0,0001 aux tests de
Wald).


In [ ]:
procédure phreg données=surv_sample;
    classe top5 / ref=first;
    modèle duration*event(0) = top5 log_officers;
    titre "Cox PH — risque de clôture par juridiction + log(dirigeants)";
exécuter;


## 9. Prédire les entités très complexes — Régression logistique

Nous définissons une entité **très complexe** comme une entité
comptant cinq dirigeants ou plus — approximativement le quartile
supérieur de la distribution — comme approximation du type de
structures densément dotées de dirigeants que les équipes de
conformité examinent en priorité. `PROC LOGISTIC` modélise
`high_complexity` en fonction de la juridiction et du nombre
d'intermédiaires.

Le cahier des charges impose un échantillonnage avec `PLOTS=NONE`
jusqu'à 5 000 lignes car le tracé ROC par défaut de `PROC LOGISTIC`
a un comportement en O(n²) à grande échelle. Nous échantillonnons
5 000 entités et utilisons `PLOTS=NONE`.


In [ ]:
procédure surveyselect données=entity_features out=ef_sample
                   method=srs sampsize=5000 seed=42;
exécuter;

données logi;
    définir ef_sample;
    longueur top5 $5;
    top5 = 'OTHER';
    si jurisdiction = 'BM' alors top5 = 'BM';
    sinon si jurisdiction = 'KY' alors top5 = 'KY';
    sinon si jurisdiction = 'VG' alors top5 = 'VG';
    sinon si jurisdiction = 'IM' alors top5 = 'IM';
    sinon si jurisdiction = 'JE' alors top5 = 'JE';
    si officer_count >= 5 alors high_complexity = 1;
    sinon high_complexity = 0;
exécuter;

procédure fréquences données=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    titre "Taux d'entités très complexes par juridiction";
exécuter;

procédure logistique données=logi descendant plots=none;
    classe top5;
    modèle high_complexity = top5 intermediary_count;
    titre "Logistique : Pr(>=5 dirigeants) ~ juridiction + intermédiaires";
exécuter;


## 10. Statistiques de synthèse consolidées

Nous suspendons le récit analytique pour capturer une synthèse
compacte via `PROC MEANS` et `PROC FREQ` des données de l'ensemble de
survie. C'est le type de tableau de tête sur lequel un analyste de
conformité ou un régulateur ouvre un rapport. Les sections qui suivent
étendent l'analyse au risque centré sur les dirigeants, aux schémas
temporels, à la concentration entre fuites, à un filtrage de sanctions
élargi et à un classement composite final des entités. Toute la sortie
est capturée dans l'unique `ODS PDF` ouvert en tête du notebook et
fermé après la Section 15.


In [ ]:
titre "Paradise Papers de l'ICIJ — Statistiques de tête";

procédure moyennes données=surv n mean std median p25 p75;
    var duration officer_count;
    titre "Synthèse durée de vie et nombre de dirigeants (n complet=12 130)";
exécuter;

procédure fréquences données=surv;
    tables top5;
    titre "Répartition par juridiction des entités survivantes";
exécuter;


## 11. Vue du risque centrée sur les dirigeants

Les Sections 2 à 10 se concentraient sur les entités. Les humains
derrière ces entités — les dirigeants — méritent la même attention.
La pratique de conformité considère qu'un dirigeant qui est
*simultanément* (a) connecté à de nombreuses entités, (b) actif dans
de nombreuses juridictions, (c) présent avec un PageRank élevé dans la
projection dirigeants-entités et (d) actif sur une longue fenêtre
temporelle constitue en soi un risque structurel.

Nous construisons une table de caractéristiques par dirigeant à partir
du graphe réel :

| Caractéristique | Définition |
|---|---|
| `degree` | Nombre d'entités où ce dirigeant détient un OFFICER_OF |
| `n_juris` | Nombre de juridictions distinctes de ces entités |
| `pagerank` | PageRank du nœud dirigeant issu de la Section 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` sur les entités du dirigeant |

Nous normalisons ensuite chaque caractéristique par min-max sur [0, 1]
et prenons une somme pondérée — 30 % degré, 30 % log-PageRank, 20 %
étendue juridictionnelle, 20 % ancienneté — comme unique **score
d'influence** composite. Les 10 premiers selon ce score font
apparaître les individus que l'équipe de recherche de l'ICIJ a
publiquement désignés comme les acteurs Appleby les plus connectés.


In [ ]:
procédure gql conn=icij out=officer_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Rattache la centralité équivalente au PageRank (issue de la sortie
   PROC NETWORK de la Section 4) via une jointure gauche sur le nom du
   dirigeant. PROC SQL gère le tri + la fusion + le coalesce en une
   seule passe — pas de chaîne DATA MERGE BY, pas de PROC SORT. */
procédure sql;
    créer table officer_feat comme
    sélectionner o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) comme pagerank
    depuis   officer_raw          comme o
    gauche joindre pagerank           comme p
      sur   o.node_id = p.node_id;
quit;

/* Normalise chaque caractéristique par min-max, construit le score
   d'influence composite, garde le top 50 par influence. PROC RANK +
   PROC SQL plutôt qu'une chaîne multi-étapes DATA. */
procédure moyennes données=officer_feat sans_impression;
    var degree pagerank n_juris tenure_years;
    sortie out=officer_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
exécuter;

données officer_scored;
    si _n_ = 1 alors définir officer_minmax;
    définir officer_feat;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    garder node_id officer_name degree pagerank
         n_juris tenure_years influence;
exécuter;

procédure sql outobs=50;
    titre "Section 11 — top 50 des dirigeants des Paradise Papers par influence composite";
    sélectionner officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    depuis   officer_scored
    ordre par influence desc;
quit;


## 12. Schémas temporels de création

En analysant `incorporation_date` côté serveur pour en extraire une
année à quatre chiffres, on peut observer comment l'activité de
création offshore a évolué dans les cinq juridictions dominantes. En
traçant le nombre annuel de nouvelles entités de 1970 à 2014 dans une
disposition de petits multiples `PROC SGPANEL`, on expose le type de
poussées liées à la législation que recherchent les analystes des
politiques publiques.

Sur les données réelles :

- L'activité des **îles Caïmans** grimpe régulièrement à partir de la
  fin des années 1990, dépasse les 400 nouvelles entités par an en
  2001 et forme un plateau jusqu'en 2014 à environ 450-510 nouvelles
  entités par an.
- Les **Bermudes** culminent autour de 2007-2013 à 210-290 par an,
  suivant de près les cycles mondiaux de levée de fonds des hedge
  funds et du capital-investissement.
- Les **îles Vierges britanniques** décollent brusquement, passant de
  ~60 nouvelles entités par an en 2005 à 200 en 2014 — une hausse de
  3,3× sur la fenêtre couverte par les Paradise Papers.
- L'**île de Man** et **Jersey** restent modestes (50-140 par an) mais
  Jersey présente un bond marqué en 2013 à 142 — le plus haut décompte
  de Jersey sur toute la fenêtre.


In [ ]:
procédure gql conn=icij out=year_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Regroupe les juridictions hors top 5 dans OTHER. */
données year_panel;
    définir year_jur;
    longueur top5 $5;
    top5 = 'OTHER';
    si jurisdiction = 'BM' alors top5 = 'BM';
    sinon si jurisdiction = 'KY' alors top5 = 'KY';
    sinon si jurisdiction = 'VG' alors top5 = 'VG';
    sinon si jurisdiction = 'IM' alors top5 = 'IM';
    sinon si jurisdiction = 'JE' alors top5 = 'JE';
exécuter;

procédure moyennes données=year_panel sans_impression nway;
    classe year top5;
    var n;
    sortie out=year_totals (supprimer=_type_ _freq_)
        sum=entities;
exécuter;

procédure sgpanel données=year_totals;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis étiquette="Année de constitution";
    rowaxis étiquette="Nouvelles entités par an";
    titre "Section 12 — créations d'entités par an, top 5 des juridictions";
exécuter;

/* Les trois plus fortes poussées annuelles sur top 5 + OTHER. */
procédure trier données=year_totals out=year_peaks;
    par descendant entities;
exécuter;

données year_peaks;
    définir year_peaks (obs=10);
exécuter;

procédure imprimer données=year_peaks;
    titre "Section 12 — dix plus fortes flambées annuelles de création (top 5 + OTHER)";
exécuter;


## 13. Comparaison entre fuites

Le graphe des Paradise Papers est scindé en interne par `sourceID` en
cinq sous-corpus, reflétant les cinq flux de sources indépendants
assemblés par l'ICIJ :

- **Paradise Papers - Appleby** — la fuite du cabinet juridique
  Appleby (l'écrasante majorité des données)
- **Paradise Papers - Malta corporate registry** — une copie fuitée du
  registre officiel des sociétés de Malte
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

En traitant chaque `sourceID` comme une « fuite », on peut confirmer
que chaque corpus se concentre dans son propre coin du monde offshore.
La fuite Appleby est très majoritairement bermudienne et caïmanaise
(un total combiné de 73 % de ses entités nommées) ; la fuite maltaise
est constituée quasi exclusivement d'entités maltaises ; la fuite
libanaise est essentiellement composée d'entités libanaises ; et ainsi
de suite. Le tableau croisé `PROC FREQ` ci-dessous rend cette
concentration visible.


In [ ]:
procédure gql conn=icij out=leak_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

procédure fréquences données=leak_raw ordre=fréquences;
    tables sourceid / out=leak_totals;
    poids n;
    titre "Section 13 — nombre d'entités par corpus sourceID";
exécuter;

procédure imprimer données=leak_totals;
    titre "Section 13 — totaux au niveau des fuites";
exécuter;

/* Le format LIST émet une ligne par cellule (fuite, juridiction) —
   tient dans la largeur du terminal au lieu du tableau croisé large
   par défaut. */
procédure trier données=leak_raw out=leak_sorted;
    par sourceid descendant n;
exécuter;

procédure imprimer données=leak_sorted (obs=30);
    titre "Section 13 — 30 principales cellules (fuite, juridiction)";
exécuter;


## 14. Enrichissement élargi des sanctions — OpenSanctions

Le filtrage de la Section 6, limité à l'OFAC-SDN, n'a renvoyé aucun
résultat en correspondance exacte. C'était un constat honnête —
l'échantillon OFAC de 500 lignes que nous avons versionné provenait en
très large majorité du programme RUSSIA-EO14024 de 2022, et les
Paradise Papers ont été compilés à partir de données dont les
enregistrements les plus récents datent de 2014.

Pour élargir le filet, nous utilisons désormais un extrait réel du jeu
de données consolidé *default* d'[OpenSanctions](https://www.opensanctions.org/) —
l'instantané du 2026-04-17 des listes de sanctions publiques
consolidées provenant de :

- U.S. OFAC Specially Designated Nationals
- cibles de gel d'avoirs du HM Treasury du Royaume-Uni
- EU Consolidated Financial Sanctions
- sanctions du Conseil de sécurité de l'ONU
- listes consolidées canadienne, belge, australienne, suisse,
  norvégienne, japonaise, néo-zélandaise et singapourienne

Le sous-ensemble versionné dans `data/opensanctions_default.csv`
contient les 18 654 enregistrements Person et Company dont le jeu de
données principal est l'une des listes de sanctions de référence
sélectionnées (les sources purement de référence, tels les catalogues
d'identifiants BIC et FIRDS, sont exclues afin que chaque résultat
porte une véritable provenance de sanction). Les alias ont été
supprimés pour maintenir le fichier sous les 2 Mo.

Comme la liste OpenSanctions est d'un ordre de grandeur plus vaste que
notre échantillon OFAC précédent, nous récupérons *une seule fois*
chaque nom de dirigeant et d'entité depuis Neo4j et effectuons la
jointure en local contre le CSV de sanctions avec `PROC SQL`. La
correspondance exacte UPCASE est robuste et évite les limites de
longueur de chaîne Cypher qui pénalisent les grandes listes IN à
nombreux jetons.


In [ ]:
/* Lit le CSV OpenSanctions versionné. Neuf lignes de commentaire
   d'en-tête plus un en-tête de colonne = firstobs=11. */
données opensanctions;
    fichier_entrée "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    longueur os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    entrée os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    si longueur(os_name_upper) >= 6;
exécuter;

procédure trier données=opensanctions nodupkey out=os_dedup;
    par os_name_upper;
exécuter;

procédure moyennes données=os_dedup n;
    titre "Enregistrements de listes de sanctions OpenSanctions chargés";
exécuter;

/* Récupère chaque nom de dirigeant + d'entité du graphe. */
procédure gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

procédure gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Correspondance exacte UPCASE — dirigeant vers partie sanctionnée. */
procédure sql;
    créer table s14_officer_hits comme
    sélectionner o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    depuis all_officers  comme o
    intérieur joindre os_dedup comme s
    sur o.officer_name_upper = s.os_name_upper;
quit;

procédure sql;
    créer table s14_entity_hits comme
    sélectionner e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    depuis all_entities comme e
    intérieur joindre os_dedup comme s
    sur e.entity_name_upper = s.os_name_upper;
quit;

procédure sql;
    sélectionner count(*) comme n_officer_hits
    depuis s14_officer_hits;
    sélectionner count(*) comme n_entity_hits
    depuis s14_entity_hits;
quit;

procédure imprimer données=s14_officer_hits;
    titre "Section 14 — dirigeants figurant sur une liste de sanctions consolidée";
exécuter;

procédure imprimer données=s14_entity_hits;
    titre "Section 14 — entités figurant sur une liste de sanctions consolidée";
exécuter;


## 15. Classement composite du risque par entité

Enfin, nous combinons les signaux structurels, juridictionnels,
relationnels et de sanctions calculés dans les sections précédentes en
un unique **score de risque** composite par entité :

| Composante | Poids | Source |
|---|---:|---|
| Nombre de dirigeants (plafonné à 50) | 0,25 | Table de caractéristiques de la Section 5 |
| log(1 + PageRank du dirigeant principal) | 0,25 | PageRank de la Section 4 + Section 11 |
| Poids de risque de la juridiction | 0,25 | Tax Justice Network CTHI 2021 (versionné) |
| Indicateur de dirigeant sanctionné | 0,25 | Résultats en correspondance exacte de la Section 14 |

Le risque juridictionnel provient du fichier versionné
`data/tax_haven_ranks.csv`, assemblé à partir de la liste de rangs
publiée dans le Corporate Tax Haven Index 2021 du Tax Justice Network.
Les rangs 1-10 sont repris directement du communiqué de presse du CTHI
2021 ; les rangs intermédiaires correspondent aux valeurs de la
méthodologie publiée par le TJN pour les autres juridictions que nous
observons dans les Paradise Papers. Les juridictions sans classement
CTHI (par ex. l'espace réservé `XX`) reçoivent un poids ≈ 0.

Le rapport ci-dessous est mis en forme par `PROC REPORT` à
l'intention d'un régulateur. Les entités en tête de liste sont celles
qui simultanément (a) sont domiciliées dans une juridiction paradis de
première page, (b) comptent de nombreux dirigeants, (c) ont un
dirigeant à PageRank élevé ET (d) ont au moins un dirigeant signalé sur
une liste de sanctions consolidée.


In [ ]:
/* Charge les rangs versionnés du Corporate Tax Haven Index 2021 du TJN.
   Huit lignes de commentaire + deux autres // plus un en-tête = firstobs=16. */
données tax_haven;
    fichier_entrée "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    longueur iso2 $5 jurisdiction_name $50;
    entrée iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
exécuter;

procédure imprimer données=tax_haven (obs=10);
    titre "Section 15 — poids de risque par juridiction (CTHI 2021)";
exécuter;

/* Caractéristiques par entité avec le nom du dirigeant principal et
   l'année de constitution. */
procédure gql conn=icij out=entity_full;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Tout ce qui doit être réuni (pagerank, poids de risque, indicateur
   de dirigeant sanctionné) est réalisé dans une unique jointure gauche
   à trois voies PROC SQL — pas de chaîne DATA MERGE BY, pas de PROC
   SORT redondants, et COALESCE fournit les valeurs de repli en ligne. */
procédure gql conn=icij out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

procédure sql;
    créer table entity_flagged comme
    sélectionner e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) comme top_officer_pr,
           coalesce(t.risk_weight, 0.0)  comme risk_weight,
           t.jurisdiction_name           comme jurisdiction_name,
           cas quand f.node_id est non nul alors 1 sinon 0 fin
               comme sanctioned_officer
    depuis   entity_full        comme e
    gauche joindre officer_scored   comme p
      sur   e.top_officer_name = p.officer_name
    gauche joindre tax_haven       comme t
      sur   e.jurisdiction     = t.iso2
    gauche joindre flagged_entity_ids comme f
      sur   e.node_id          = f.node_id;
quit;

/* Risque composite : normalise par min-max les caractéristiques
   continues, puis les pondère ensemble. PROC MEANS + une seule étape
   DATA convient pour la normalisation — c'est du SAS idiomatique. */
procédure moyennes données=entity_flagged sans_impression;
    var top_officer_pr;
    sortie out=pr_max_ds max=pr_max;
exécuter;

données entity_score;
    si _n_ = 1 alors définir pr_max_ds;
    définir entity_flagged;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    garder node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
exécuter;

/* Distribution du risque sur la population complète de 24 957 entités. */
procédure moyennes données=entity_score n min mean max;
    var risk officer_count risk_weight;
    titre "Section 15 — distribution du risque sur l'ensemble des entités";
exécuter;

/* Top 25 des entités par risque composite. Le OUTOBS= de PROC SQL
   remplace un couple PROC SORT + étape DATA obs=. */
procédure sql outobs=25;
    titre "Section 15 — top 25 des entités par risque composite (noms)";
    sélectionner entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    depuis   entity_score
    ordre par risk desc;
quit;

/* Fait ressortir séparément les entités liées à un dirigeant sanctionné. */
procédure sql;
    titre "Section 15 — entités ayant au moins un dirigeant sanctionné";
    sélectionner entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    depuis   entity_score
    où  sanctioned_officer = 1
    ordre par risk desc;
quit;


## 16. Classification des juridictions conduit vs puits

**Référence :** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo et al. partitionnent le graphe mondial de propriété
des sociétés en « puits-OFC » — où la valeur des sociétés se termine :
BM, KY, BVI, JE, IM — et « conduits-OFC » — à travers lesquels la
valeur transite : NL, UK, CH, SG, IE. Le graphe des Paradise Papers a
une population différente (surtout des entités domiciliées chez
Appleby), mais la même distinction structurelle devrait séparer les
juridictions où les dirigeants se regroupent et où les adresses se
terminent de celles qui canalisent principalement les capitaux.

Pour chaque juridiction, nous calculons cinq caractéristiques
structurelles directement à partir du graphe en direct :

| Caractéristique | Signal de |
|---|---|
| `log(n_entity)` | taille absolue de la présence offshore de la juridiction |
| `avg_officers` | densité de dirigeants par entité (les juridictions puits portent plus de dirigeants par entité) |
| `avg_xborder_off` | nombre moyen de dirigeants dont le propre pays diffère de la juridiction de l'entité (indicateur de conduit) |
| `intermediary_share` | part des entités ayant un lien d'intermédiaire CONNECTED_TO |
| `address_share` | part des entités ayant un lien REGISTERED_ADDRESS (indicateur de puits) |

Nous standardisons en scores z, puis appliquons la classification
hiérarchique à variance minimale de Ward. `PROC TREE` affiche le
dendrogramme. À noter que les nœuds Intermediary des Paradise Papers se
connectent aux nœuds Entity via `CONNECTED_TO` — et non
`INTERMEDIARY_OF`, qui existe dans le schéma mais ne porte aucune
donnée pour cette fuite — nous utilisons donc `CONNECTED_TO` ici.


In [ ]:
/* Récupère du graphe en direct les caractéristiques structurelles par
   juridiction. */
procédure gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Ne garde que les juridictions ayant au moins dix entités pour que les
   scores z standardisés soient significatifs.  La fuite des Paradise
   Papers compte 44 juridictions au total ; 23 atteignent ce seuil. */
données s16_filtered;
    définir s16_raw;
    si n_entity >= 10;
    log_n_entity = log(n_entity);
exécuter;

procédure moyennes données=s16_filtered sans_impression;
    var log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    sortie out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
exécuter;

données s16_std;
    si _n_ = 1 alors définir s16_stats;
    définir s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    garder jurisdiction z1 z2 z3 z4 z5;
exécuter;

procédure imprimer données=s16_std;
    titre "Section 16 — caractéristiques standardisées des juridictions";
exécuter;

/* Classification hiérarchique à variance minimale de Ward. */
procédure grappe données=s16_std method=ward outtree=s16_tree;
    var z1 z2 z3 z4 z5;
    id jurisdiction;
    titre "Section 16 — classification de Ward (typologie Garcia-Bernardo 2017)";
exécuter;

/* Affiche le dendrogramme. */
procédure tree données=s16_tree horizontal;
    id jurisdiction;
    titre "Section 16 — dendrogramme conduit vs puits, Paradise Papers";
exécuter;


## 17. Composantes principales des rôles réseau des dirigeants

**Référence :** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Voir aussi Newman M E J, *Networks: An Introduction* (Oxford, 2010),
chapitre 7.

Les dirigeants du graphe des Paradise Papers jouent des rôles
structurellement différents. Certains se trouvent au centre d'une
vaste grappe d'entités liées ; d'autres relient des grappes autrement
séparées (des courtiers, au sens de Burt/Borgatti) ; quelques-uns
forment le cœur dense du réseau d'initiés d'une juridiction
particulière. Quatre métriques de graphe capturent ces rôles
distincts :

| Métrique | Capte |
|---|---|
| `degree` | Nombre d'arêtes sortantes `OFFICER_OF` — étendue de l'affiliation |
| `betweenness` | Fraction des plus courts chemins passant par le dirigeant — **intermédiation** |
| `kcore` | Plus grand k pour lequel le dirigeant appartient à un sous-graphe k-connexe — **enracinement dans le cœur** |
| `pagerank` | Score de type vecteur propre issu de la même projection — **influence par des partenaires influents** |

Nous calculons les quatre métriques sur la projection non orientée
complète `(Officer)—[OFFICER_OF]—(Entity)` via la bibliothèque Graph
Data Science de Neo4j, restreignons aux 5 000 dirigeants de plus haut
degré et exécutons `PROC PRINCOMP` sur les métriques transformées en
logarithme.

L'ACP compresse les quatre métriques corrélées en axes orthogonaux
dont les charges relatives nous indiquent quels rôles se regroupent
empiriquement et lesquels sont structurellement distincts.

*Note sur le coefficient de clustering local :* Garcia-Bernardo et al.
incluent le coefficient de clustering local comme cinquième métrique.
La projection `(Officer)—[:OFFICER_OF]—(Entity)` des Paradise Papers
est strictement biparti, donc aucun triangle ne peut exister — tout
coefficient de clustering local vaut 0. Nous l'écartons de l'ensemble
des métriques.


In [ ]:
/* PROC NETWORK récupère un sous-graphe des 5 000 principaux dirigeants
   via un MATCH en lecture seule et calcule le degré, la centralité de
   vecteur propre et le k-core en mémoire. Remplace un précédent
   gds.graph.project + quatre appels gds.<algorithme>.stream — ce
   schéma exige une étape de projection GDS en mode écriture que le
   service step-neo4j en lecture seule de la plateforme refuse.

   La centralité d'intermédiarité est intentionnellement omise :
   NetworkX calcule l'intermédiarité exacte en O(V·E) qui domine le
   temps d'exécution sur ce sous-graphe (5 000 dirigeants × ~78 000
   arêtes). L'ACP conserve tout de même trois axes orthogonaux — degré
   (proéminence locale), centralité de vecteur propre (influence
   globale) et k-core (cohésion structurelle) — de quoi séparer les
   rôles de dirigeants pour l'interprétation de tête. */
procédure network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar depuis=a_node_id jusqu_à=b_node_id;
    centrality degree eigen=unweight;
    core;
exécuter;

/* Récupère les identifiants/noms des nœuds dirigeants pour le filtrage. */
procédure gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Filtre sur les lignes de dirigeants. La centralité de vecteur propre
   tient lieu de PageRank — voir le commentaire de la Section 4. */
procédure sql;
    créer table s17_metrics comme
    sélectionner n.node                          comme node_id,
           o.officer_name                  comme officer_name,
           n.centr_degree                  comme degree,
           coalesce(n.core_out, 0)         comme kcore,
           coalesce(n.centr_eigen_unwt, 0) comme pagerank
    depuis s17_metrics_full comme n
    intérieur joindre all_officer_names comme o sur n.node = o.node_id
    ordre par n.centr_degree desc;
quit;

données s17_metrics;
    définir s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    garder node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
exécuter;

procédure imprimer données=s17_metrics (obs=10);
    titre "Section 17 — top 10 des dirigeants par degré (métriques brutes + log)";
exécuter;

procédure moyennes données=s17_metrics n mean std min max;
    var log_degree log_pr k_val;
    titre "Section 17 — synthèse des métriques transformées en logarithme";
exécuter;

procédure princomp données=s17_metrics out=s17_scores;
    var log_degree log_pr k_val;
    titre "Section 17 — ACP des rôles réseau des dirigeants";
exécuter;

procédure sgplot données=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis étiquette="PC1 (axe d'influence globale)";
    yaxis étiquette="PC2 (intermédiation vs enracinement dans le cœur)";
    titre "Section 17 — dirigeants dans l'espace 2D des rôles en composantes principales";
exécuter;


## 18. Analyse d'intervention ARIMA sur les taux de création

**Référence :** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Appliqué aux fuites offshore par Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021).

Le nombre annuel de nouvelles entités dans le graphe des Paradise
Papers est une série de croissance quasi monotone de 1970 (36 entités)
à 2007 (1 595 entités, le pic), suivie d'une chute en 2008-2009 puis
d'un plateau plus lent jusqu'en 2014 (fin de la couverture de la
fuite).

Nous appliquons l'analyse d'intervention classique de Box-Tiao pour
tester si deux événements réels ont laissé des signatures mesurables
dans la série de création :

- **Marche 2009** — la répression des paradis fiscaux du sommet du G20
  de Londres (avril 2009), qui a coïncidé avec la crise financière.
- **Marche 2014** — le FATCA américain (Foreign Account Tax Compliance
  Act) est entré en vigueur le 1er juillet 2014.

La réponse est `log(n)`, de sorte qu'un coefficient d'intervention de
-0,30 correspond à une baisse d'environ 26 % du taux annuel de
création. Nous ajustons `ARIMA(1,0,0)` — le modèle autorégressif AR(1)
sur la série fortement corrélée — avec les deux indicatrices de marche
comme variables exogènes `INPUT=`.

L'hypothèse nulle est qu'aucune des deux marches ne produit de
décalage mesurable une fois la tendance AR(1) prise en compte. Les
méthodologies publiées divergent sur l'interprétation d'un non-rejet :
cela peut signifier que l'intervention n'a eu aucun effet, ou que
l'autocorrélation AR(1) absorbe le signal de l'intervention.


In [ ]:
procédure gql conn=icij out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Construit le jeu de données de la série d'intervention.  Deux
   indicatrices de marche : step_2009 = I{year >= 2009} capture le
   changement de régime pré-annonce G20 Londres/FATCA ; step_2014 =
   I{year >= 2014} capture le choc de la date d'entrée en vigueur du
   FATCA.  La réponse est log(n), donc les estimations de coefficients
   se lisent comme des effets proportionnels. */
données s18_ts;
    définir year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
exécuter;

procédure imprimer données=s18_ts;
    titre "Section 18 — créations annuelles de nouvelles entités + indicatrices d'intervention";
exécuter;

procédure sgplot données=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x étiquette="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x étiquette="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis étiquette="Année de constitution";
    yaxis étiquette="Nouvelles entités par an";
    titre "Section 18 — créations annuelles des Paradise Papers, 1970-2014";
exécuter;

/* Identifie le modèle, puis estime ARIMA(1,0,0) avec les deux entrées
   de marche.  Le CROSSCORR= de PROC ARIMA enregistre les variables
   exogènes à l'étape IDENTIFY afin qu'elles soient disponibles pour
   ESTIMATE INPUT=. */
procédure arima données=s18_ts;
    identify var=log_n crosscorr=(step_2009 step_2014) nlag=8;
    estimation p=1 entrée=(step_2009 step_2014);
    titre "Section 18 — ARIMA(1,0,0) avec marches G20-2009 et FATCA-2014";
exécuter;
quit;


## 19. Modèle de comptage à inflation de zéros de l'exposition aux sanctions

**Référence :** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2e édition, Cambridge University Press (2013), chapitre 4.
Modèles à inflation de zéros : Lambert D. *Zero-inflated Poisson
regression with an application to defects in manufacturing.*
Technometrics 34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

La Section 14 n'a trouvé que **cinq** entités des Paradise Papers ayant
au moins un dirigeant sur une liste de sanctions consolidée — un
événement extrêmement rare. Une régression de Poisson ou binomiale
négative standard sur `sanctioned_count` par entité s'ajusterait mal
car **99,98 %** des entités valent zéro. Le modèle binomial négatif à
inflation de zéros (ZINB) gère cela en scindant l'ajustement en deux
parties :

1. Un modèle logistique déterminant si l'entité appartient à une
   classe de « zéro structurel » (aucune exposition possible aux
   sanctions).
2. Un modèle binomial négatif pour le comptage sur le reste.

Avec seulement 5 événements positifs sur 21 538 entités, la
vraisemblance ZINB est dégénérée — les deux parties s'effondrent. Cet
échec est une **propriété honnête des données**, pas de la procédure.
Nous exécutons tout de même l'ajustement ZINB pour montrer que
l'outillage de régression fonctionne de bout en bout, puis nous nous
rabattons sur une régression logistique binaire ordinaire de
`has_sanctioned` (indicateur de `sanctioned_count > 0`). La régression
logistique identifie un effet net : **chaque unité logarithmique
supplémentaire du nombre de dirigeants multiplie par environ 3,1 la
cote d'avoir au moins un dirigeant sanctionné** (p = 0,002).

Covariables :

- `top5` — variable de classe à 6 niveaux (BM, KY, VG, IM, JE, OTHER),
  catégorie de référence OTHER.
- `log_n_off` — `log(officer_count)`, le prédicteur de taille dominant.


In [ ]:
/* Récupère du graphe en direct le nombre de dirigeants sanctionnés par
   entité. */
procédure gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

données s19;
    définir s19_raw;
    si officer_count >= 1;
    longueur top5 $5;
    top5 = 'OTHER';
    si jurisdiction = 'BM' alors top5 = 'BM';
    sinon si jurisdiction = 'KY' alors top5 = 'KY';
    sinon si jurisdiction = 'VG' alors top5 = 'VG';
    sinon si jurisdiction = 'IM' alors top5 = 'IM';
    sinon si jurisdiction = 'JE' alors top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
exécuter;

procédure fréquences données=s19;
    tables top5 sanctioned_count has_sanctioned;
    titre "Section 19 — distribution de la variable réponse (très chargée en zéros)";
exécuter;

/* ZINB d'abord — attendu dégénéré sur une série de 5 événements. */
procédure genmod données=s19;
    classe top5;
    modèle sanctioned_count = top5 log_n_off / dist=zinb link=log;
    titre "Section 19 — modèle de comptage ZINB (dégénéré sur 5 événements)";
exécuter;

/* Repli : logistique binaire sur has_sanctioned.  Interprétable. */
procédure logistique données=s19 descendant plots=none;
    classe top5;
    modèle has_sanctioned = top5 log_n_off;
    titre "Section 19 — régression logistique (repli has-sanctioned)";
exécuter;


## 20. Modèle de Poisson à effets mixtes du taux de création

**Référence :** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Comptage de panel classique : Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

La Section 18 ajustait un ARIMA univarié à la série agrégée de
création. Ici, nous exploitons la dimension **panel** : une ligne par
cellule juridiction-année, en ajustant un modèle linéaire généralisé
mixte (GLMM) de Poisson avec une tendance annuelle linéaire fixe, une
indicatrice de marche FATCA et un **intercept aléatoire par
juridiction**. Cela sépare l'effet de tendance commune du niveau
propre à chaque juridiction.

Restriction du panel : les 10 juridictions dont le **nombre annuel
moyen** est >=5 sur 1990-2014 (203 cellules au total). Les juridictions
plus petites, comptant de nombreuses années à zéro, déstabiliseraient
un ajustement de Poisson.

`PROC GLIMMIX` avec `DIST=POISSON LINK=LOG` et
`RANDOM INTERCEPT / SUBJECT=jurisdiction` produit à la fois les effets
fixes au niveau de la population (tendance annuelle, décalage FATCA) et
la composante de variance inter-juridictions. La variance de
l'intercept aléatoire nous indique *dans quelle mesure les taux de
création diffèrent entre juridictions après prise en compte de la
tendance temporelle commune* — un score d'hétérogénéité structurelle
pour la population des centres financiers offshore.


In [ ]:
/* Jeu de données de panel : cellules juridiction x année de 1990-2014. */
procédure gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Garde les juridictions ayant un nombre annuel moyen >= 5. */
procédure sql;
    créer table s20_keep_jur comme
    sélectionner jurisdiction, avg(n) comme avg_n
    depuis s20_raw
    groupe par jurisdiction
    ayant avg(n) >= 5;
quit;

procédure sql;
    créer table s20 comme
    sélectionner r.*,
           r.year - 2002 comme year_c,
           cas quand r.year >= 2014 alors 1 sinon 0 fin comme fatca
    depuis s20_raw r
    intérieur joindre s20_keep_jur k
    sur r.jurisdiction = k.jurisdiction;
quit;

procédure fréquences données=s20;
    tables jurisdiction;
    titre "Section 20 — juridictions retenues dans le panel";
exécuter;

/* GLMM de Poisson à effets mixtes : tendance annuelle fixe + marche
   FATCA, intercept aléatoire par juridiction. */
procédure glimmix données=s20;
    classe jurisdiction;
    modèle n = year_c fatca / dist=poisson link=log solution;
    aléatoire intercept / subject=jurisdiction;
    titre "Section 20 — modèle de Poisson mixte du taux de création";
exécuter;

/* Le classement des effets d'intercept aléatoire ferait ressortir les
   juridictions qui surperforment ou sous-performent systématiquement la
   tendance commune.  L'instruction SOLUTION de PROC GLIMMIX les affiche
   dans la sortie par défaut ci-dessus — nous laissons le classement au
   lecteur. */


In [ ]:
/* Ferme le PDF du rapport et libère la bibliothèque Neo4j. */
ods pdf close;

librairie icij clear;


## Reproductibilité et provenance

- **Source des données de graphe :** base de recherche ICIJ *Offshore
  Leaks*, jeu de données *Paradise Papers*, publié pour la première
  fois en novembre 2017. Disponible à
  <https://offshoreleaks.icij.org/pages/database>. Chargée dans le
  service partagé `step-neo4j` de la plateforme (Neo4j 5.26 Community
  Edition, en lecture seule au niveau du serveur) via le dump public
  amont à `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **Données OFAC SDN :** export CSV public de la liste des Specially
  Designated Nationals de l'OFAC du Trésor américain, récupéré depuis
  l'API de prévisualisation publique du Trésor en avril 2026. Le
  fichier `data/ofac_sdn.csv` contient 500 lignes sélectionnées parmi
  les cinq plus grands programmes OFAC par nombre d'entrées. Utilisé
  pour le filtrage en deux étapes de la Section 6.
- **Données OpenSanctions :** instantané du jeu de données consolidé
  *default* d'OpenSanctions du 2026-04-17, téléchargé depuis
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  Le fichier versionné `data/opensanctions_default.csv` contient les
  18 654 lignes de schéma Person et Company dont le jeu de données
  principal est l'une des listes de sanctions consolidées OFAC SDN, HM
  Treasury du Royaume-Uni, sanctions financières de l'UE, Conseil de
  sécurité de l'ONU, canadienne, belge, australienne, suisse ou d'un
  autre pays. Les alias ont été supprimés pour maintenir le fichier
  sous les 2 Mo. Licence : ODbL 1.0 (OpenSanctions). Utilisé pour
  l'enrichissement de la Section 14.
- **Rangs des paradis fiscaux :** classements publiés du *Corporate Tax
  Haven Index 2021* du Tax Justice Network, compilés à partir de la
  page d'accueil de l'indice `https://cthi.taxjustice.net` et du
  communiqué de presse de mars 2021 à
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  Le fichier versionné `data/tax_haven_ranks.csv` liste les
  juridictions présentes dans les Paradise Papers avec leur rang CTHI
  et un poids de risque dérivé dans `[0, 1]`. Licence : CC BY-SA 4.0
  (Tax Justice Network). Utilisé pour le classement composite de la
  Section 15.
- **Algorithmes de graphe :** détection de communautés par Louvain et
  centralité de vecteur propre (l'analogue interne le plus proche de
  PageRank) calculées en mémoire par `PROC NETWORK` sur des listes
  d'arêtes récupérées via un Cypher en lecture seule. Le service
  partagé `step-neo4j` de la plateforme est en lecture seule au niveau
  du serveur, si bien que tous les algorithmes de graphe s'exécutent
  dans le pod du workspace plutôt que via des opérations d'écriture
  Neo4j Graph Data Science.
- **Méthodes statistiques :** `PROC LIFETEST` utilise l'estimateur de
  Kaplan-Meier avec les tests stratifiés du log-rank, de Wilcoxon et
  de Tarone-Ware. `PROC PHREG` ajuste le modèle à risques
  proportionnels de Cox par la méthode de Breslow pour les ex æquo, via
  le wrapper Python/statsmodels. `PROC LOGISTIC` ajuste une régression
  logistique binaire par maximum de vraisemblance.
- **Méthodes de composition du risque :** le score d'influence
  composite de la Section 11 normalise le degré, le log-PageRank,
  l'étendue juridictionnelle et l'ancienneté en années sur `[0, 1]`
  puis somme avec des poids fixes (30/30/20/20). Le score de risque
  composite par entité de la Section 15 normalise le nombre de
  dirigeants plafonné, le log-PageRank, le poids de risque CTHI et un
  indicateur binaire de dirigeant sanctionné, puis somme avec des poids
  égaux de 0,25 chacun.

L'analyse complète est reproductible à partir du script de test de
fumée de ce dossier : `./smoke.jenner`. Son exécution de bout en bout
contre le service partagé `step-neo4j` (avec `JENNER_NEO4J_HOST` et
`JENNER_NEO4J_PASS` définis, ce que la plateforme fait pour vous dans
tout pod de workspace) prend environ deux minutes et vérifie que
chaque requête et chaque PROC — y compris les cinq nouvelles sections
ajoutées à la chaîne existante — renvoient la sortie attendue sur le
graphe réel de 163 435 nœuds.
